# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

Before you start, install the some of the required packages. 

In [1]:
# Not needed for this local project: dependencies are installed from requirements.txt
# into the local .venv (Python 3.14). This starter cell targeted the Udacity cloud
# workspace and an old kaggle version, so it is commented out.
# !python -m pip install kaggle==1.6.12

In [2]:
# Not needed for this local project. The original `--target=/workspace` path is the
# Udacity cloud workspace (nonexistent locally), and numpy==1.24.3 will not build on
# Python 3.14. ucimlrepo/numpy are already installed in the local .venv, so this is
# commented out.
# !pip install --target=/workspace ucimlrepo numpy==1.24.3

**Note:** Not applicable here — the two install cells above are commented out because dependencies are installed from `requirements.txt` into the local `.venv` (Python 3.14). No kernel restart is needed. *(Originally: "Restart the kernel to use updated package(s).")*

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement
In 2-4 sentences, explain the kind of problem you want to look at and the datasets you will be wrangling for this project.

In the Udacity Master's Degree in AI, I would like to investigate the possibility of using AI to predict the infrared spectral lines of complex aromatic molecules, so that they can be used to identify them in observations from space telescopes. To prepare for this work, I would like to familiarize myself with some of the world's molecular datasets, in this project's case: the NASA Ames PAH IR Spectroscopic Database on the one hand, and PubChem on the other. I would like to try to connect them if that is at all possible. For the sake of this Data Wrangling project, the research question could be:  

*How does the number of distinct infrared emission lines vary with an aromatic molecule's size (molecular weight or carbon count), and do neutral and ionized aromatics differ in this respect?*

For this project, the two datasets are sourced from the following:

* **NASA Ames PAH IR Spectroscopic Database (PAHdb)** — infrared spectra of polycyclic aromatic hydrocarbons (theoretical library): https://www.astrochemistry.org/pahdb/
* **PubChem** — chemical structure and molecular properties, from the NCBI/NIH database: https://pubchem.ncbi.nlm.nih.gov/ (programmatic access via PUG REST: https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest)

*The starter notebook originally suggested these general dataset directories: [Google Dataset Search](https://datasetsearch.research.google.com/), [data.gov](https://data.gov/), and the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/index.php).*

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code). 

Load the dataset programmtically into this notebook.

#### **Dataset 1: NASA Ames PAH IR Spectroscopic Database (PAHdb)**

Type: **XML file** (the complete *Theoretical* library, version 4.00).

Method: **Manual download ("Downloading files").** PAHdb distributes its libraries through an email-gated web form ([download page](https://www.astrochemistry.org/pahdb/theoretical/4.00/download/view)) — you select a library, choose *Download as XML*, provide an email address, and receive a link by email. Because there is no public static URL to download from, a manual download is the appropriate gathering method. The delivered file was saved to `data/raw/pahdb-complete-theoretical-v4.00.xml` (≈480 MB, 10,749 species). The code below programmatically loads and parses that XML into a DataFrame.

I picked PAHdb because it is the world's foremost collection of infrared (IR) spectra of polycyclic aromatic hydrocarbons (PAHs), which is exactly the spectral information needed for the research question, and it provides the ionization state that lets us compare neutral versus ionized molecules.

Dataset variables (per molecule):

*   **uid**: unique PAHdb identifier for each computed species.
*   **formula**: molecular formula (e.g., `C6H6O`) — the elemental composition, and the source of the carbon count used as a size measure.
*   **charge**: ionization state (`0` = neutral, `1`/`2`/`3` = cations, `-1` = anion) — used to compare neutral vs. ionized aromatics.
*   **weight**: molecular weight (g/mol) — a measure of molecular size.
*   **symmetry**: point-group symmetry label of the molecule.
*   **method**: the quantum-chemistry method used for the computation (e.g., `RB3LYP`).
*   **frequencies** / **intensities**: the IR emission lines. Each molecule has many vibrational modes, so these are stored as multiple values per cell (a deliberate tidiness issue to resolve later). The **number of distinct emission lines** (the count of modes) is the key variable for the research question.

There are 10,749 species (well over the 500-sample minimum) and many variables per species.

In [3]:
# --- Dataset 1: NASA Ames PAHdb (manual download, XML) ---
# The complete Theoretical library v4.00 was downloaded manually as XML (email-gated)
# from https://www.astrochemistry.org/pahdb/theoretical/4.00/download/view and saved
# to data/raw/. Here we stream-parse that XML into one row per molecule.
import os
import xml.etree.ElementTree as ET
import pandas as pd

RAW_DIR = os.path.join("data", "raw")
PAHDB_XML = os.path.join(RAW_DIR, "pahdb-complete-theoretical-v4.00.xml")
PAHDB_EXTRACT = os.path.join(RAW_DIR, "pahdb_theoretical_extracted.csv")

# The PAHdb XML uses a default namespace, so element names are prefixed with it.
NS = "{http://www.astrochemistry.org/pahdb/theoretical}"


def parse_pahdb(xml_path):
    """Stream-parse the ~480 MB PAHdb XML into one row per molecule.

    iterparse + elem.clear() keeps memory low. The vibrational modes (the IR
    emission lines) are kept as ';'-joined strings so each molecule stays on a
    single row; this intentionally leaves multiple values in one cell, which is
    tidied later with an explode.
    """
    records = []
    for _, elem in ET.iterparse(xml_path, events=("end",)):
        if elem.tag.split("}", 1)[-1] != "specie":
            continue
        # Collect the frequency/intensity of every vibrational mode.
        freqs, intens = [], []
        transitions = elem.find(f"{NS}transitions")
        if transitions is not None:
            for mode in transitions.findall(f"{NS}mode"):
                freq = mode.find(f"{NS}frequency")
                inten = mode.find(f"{NS}intensity")
                if freq is not None and inten is not None:
                    freqs.append(freq.text)
                    intens.append(inten.text)

        def get(name):
            child = elem.find(f"{NS}{name}")
            return child.text if child is not None else None

        records.append({
            "uid": elem.get("uid"),
            "formula": get("formula"),
            "charge": get("charge"),
            "weight": get("weight"),
            "symmetry": get("symmetry"),
            "method": get("method"),
            "frequencies": ";".join(freqs),
            "intensities": ";".join(intens),
        })
        elem.clear()  # release the parsed subtree to keep memory low
    return pd.DataFrame(records)


# Parse the XML once, then cache a compact extract for fast re-runs.
if os.path.exists(PAHDB_EXTRACT):
    df_pahdb = pd.read_csv(PAHDB_EXTRACT, dtype=str)
    print(f"Loaded cached extract: {PAHDB_EXTRACT}")
elif os.path.exists(PAHDB_XML):
    df_pahdb = parse_pahdb(PAHDB_XML)
    df_pahdb.to_csv(PAHDB_EXTRACT, index=False)
    print(f"Parsed {PAHDB_XML} -> cached to {PAHDB_EXTRACT}")
else:
    raise FileNotFoundError(
        "PAHdb XML not found. Download the complete Theoretical library v4.00 as XML "
        "from https://www.astrochemistry.org/pahdb/theoretical/4.00/download/view "
        f"(email-gated) and save it to {PAHDB_XML}."
    )

print(f"df_pahdb shape: {df_pahdb.shape}")
df_pahdb.head()

Loaded cached extract: data\raw\pahdb_theoretical_extracted.csv
df_pahdb shape: (10749, 8)


,uid,formula,charge,weight,symmetry,method,frequencies,intensities
0,430,C6H6O,0,94.04186000,1-A',RB3LYP,68.22150000;266.11240000;438.55110000;445.6087...,6.57630000;0.00280000;0.00300000;8.90640000;11...
1,428,C6H6O,0,94.04186000,1-A',RB3LYP,228.83890000;351.24410000;398.38840000;413.898...,0.63210000;123.24690000;9.97170000;0.16920000;...
2,431,C6H6O+,1,94.04186000,2-A,UB3LYP,112.16990000;245.00450000;400.63220000;423.474...,7.54030000;4.52050000;2.94200000;15.97010000;2...
3,429,C6H6O+,1,94.04186000,"2-A""",UB3LYP,183.67720000;356.11560000;405.28100000;429.512...,1.12900000;3.23460000;10.24320000;0.47810000;0...
4,432,C6H7O+,1,95.04968500,1-A',RB3LYP,136.62370000;261.89450000;411.13890000;432.735...,4.28700000;5.87670000;7.88840000;0.58710000;42...


#### Dataset 2

Type: *FILL IN* (e.g., CSV File.)

Method: *FILL IN* (e.g., The data was gathered using the "API" method from Y source.)

Dataset variables:

*   *Variable 1 FILL IN* (e.g., H_MEAN: Mean hourly wage)
*   *Variable 2 FILL IN*

In [ ]:
#FILL IN 2nd data gathering and loading method

Optional data storing step: You may save your raw dataset files to the local data store before moving to the next step.

In [ ]:
#Optional: store the raw data in your local data store

## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

### Quality Issue 1:

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

### Quality Issue 2:

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

### Tidiness Issue 1:

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

### Tidiness Issue 2: 

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
# FILL IN - Make copies of the datasets to ensure the raw dataframes 
# are not impacted

### **Quality Issue 1: FILL IN**

In [ ]:
# FILL IN - Apply the cleaning strategy

In [ ]:
# FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Quality Issue 2: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Tidiness Issue 1: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Tidiness Issue 2: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Remove unnecessary variables and combine datasets**

Depending on the datasets, you can also peform the combination before the cleaning steps.

In [ ]:
#FILL IN - Remove unnecessary variables and combine datasets

## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
#FILL IN - saving data

## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

*Research question:* FILL IN from answer to Step 1

In [ ]:
#Visual 1 - FILL IN

*Answer to research question:* FILL IN

In [ ]:
#Visual 2 - FILL IN

*Answer to research question:* FILL IN

### **5.2:** Reflection
In 2-4 sentences, if you had more time to complete the project, what actions would you take? For example, which data quality and structural issues would you look into further, and what research questions would you further explore?

*Answer:* FILL IN